# VGG16 Image Model

In [ ]:
# Import necessary libraries for model building and evaluation
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.applications import VGG16
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [ ]:
# To run this notebook, ensure that you first run
# IMAGE_MODEL_CREATE_TRAIN_VAL_TEST_DATASETS.ipynb
# to create the datasets. Copy the printed paths and paste them below.

# Define paths for training, validation, and test datasets
train_data_dir = '/Users/et/code/Lucia-Cordero/ReefSight-Project/raw_data/image_data/train_validation_test/train'
val_data_dir = '/Users/et/code/Lucia-Cordero/ReefSight-Project/raw_data/image_data/train_validation_test/val'
test_data_dir = '/Users/et/code/Lucia-Cordero/ReefSight-Project/raw_data/image_data/train_validation_test/test'

# Define the path where the model will be saved
model_path = '/Users/et/code/Lucia-Cordero/ReefSight-Project/models'

In [ ]:
# Constants
BATCH_SIZE = 16
SEED = 42
IMAGE_SIZE = (224, 224)
INPUT_SHAPE = IMAGE_SIZE + (3,)  # Adding the channel dimension
NUM_CLASSES = 2
LEARNING_RATE = 0.001
EPOCHS = 1000
L2_REGULARIZATION_FACTOR = 0.1
THRESHOLD = 0.5

# Load datasets
train_ds = image_dataset_from_directory(
  train_data_dir,
  labels="inferred",
  label_mode="binary",
  seed=SEED,
  image_size=IMAGE_SIZE,
  batch_size=BATCH_SIZE
  )

val_ds = image_dataset_from_directory(
  val_data_dir,
  labels="inferred",
  label_mode="binary",
  seed=SEED,
  image_size=IMAGE_SIZE,
  batch_size=BATCH_SIZE
  )

test_ds = image_dataset_from_directory(
  test_data_dir,
  labels="inferred",
  label_mode="binary",
  seed=SEED,
  image_size=IMAGE_SIZE,
  batch_size=BATCH_SIZE
  )

print("Class names train dataset:", train_ds.class_names)
print("Class names validation dataset:", val_ds.class_names)
print("Class names test dataset:", test_ds.class_names)

# Built and compile model
def create_vgg16_model(input_shape, num_classes=NUM_CLASSES, learning_rate=LEARNING_RATE, l2_reg=L2_REGULARIZATION_FACTOR):
    """Builds a VGG16 model with transfer learning and regularization.
    Args:
        input_shape (tuple): Shape of the input images.
        num_classes (int): Number of output classes.
        learning_rate (float): Learning rate for the optimizer.
        l2_reg (float): L2 regularization factor.

    Returns:
        model: Compiled Keras Sequential model.
    """
    # Build the model
    model = models.Sequential()

    # Add data augmentation layers
    model.add(layers.RandomFlip(mode="horizontal_and_vertical", input_shape=INPUT_SHAPE))
    model.add(layers.RandomRotation(0.2))
    model.add(layers.RandomZoom(0.2))

    # Normalize pixel values to [0, 1]
    model.add(layers.Rescaling(1./255, input_shape=INPUT_SHAPE))

    # Load VGG16 without the top layers (classifier) and with pre-trained weights
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)

    # Unfreeze the last few layers for fine-tuning
    for layer in base_model.layers[-4:]:
        layer.trainable = True
    for layer in base_model.layers[:-4]:
        layer.trainable = False

    # Add the base model
    model.add(base_model)

    # Use Global Average Pooling
    model.add(layers.GlobalAveragePooling2D())

    # Fully connected layers with L2 regularization
    model.add(layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)))
    model.add(layers.Dropout(0.2))

    # Output layer
    model.add(layers.Dense(1, activation='sigmoid'))  # Sigmoid for binary classification

    # Compile the model
    adam = Adam(learning_rate=LEARNING_RATE)
    model.compile(optimizer=adam, loss='binary_crossentropy', metrics=['accuracy'])

    return model

# Create and display the model summary
vgg16_model = create_vgg16_model(INPUT_SHAPE, NUM_CLASSES, LEARNING_RATE)
vgg16_model.summary()

# Configure callbacks
callbacks = [
  EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
  ModelCheckpoint(os.path.join(model_path, 'VGG16_image_model.keras'), monitor='val_accuracy', save_best_only=True),
  ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=4, verbose=1),
]

## Train using datasets
history = vgg16_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

# Plot training history
def plot_history(history):
    """Plots the training history for loss and accuracy."""

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    ax[0].set_title('Loss')
    ax[0].plot(history.epoch, history.history['loss'], label='Train Loss')
    ax[0].plot(history.epoch, history.history['val_loss'], label='Validation Loss')
    ax[0].legend()

    ax[1].set_title('Accuracy')
    ax[1].plot(history.epoch, history.history['accuracy'], label='Train Accuracy')
    ax[1].plot(history.epoch, history.history['val_accuracy'], label='Validation Accuracy')
    ax[1].legend()

plot_history(history)

# Evaluate the model on the validation dataset
val_loss, val_accuracy = vgg16_model.evaluate(val_ds)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")

# Evaluate the model on the test dataset
test_loss, test_accuracy = vgg16_model.evaluate(test_ds)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

# Plot confusion matrix
y_true = []
y_pred = []

# Iterate through the test dataset to collect predictions
for images, labels in test_ds:
    y_true.extend(labels.numpy())
    predictions = vgg16_model.predict(images)
    # For binary classification, use a threshold of 0.5 to determine class
    predicted_class = (predictions > THRESHOLD).astype(int)  # Convert probabilities to class labels
    y_pred.extend(predicted_class)

# Create confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=test_ds.class_names)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()